In [88]:
import pandas as pd
df = pd.read_json("/ssdscratch/byuan48/efficient_reasoning/results/Llama-3.2-3B-Instruct-Wmajority-16.jsonl", lines=True)

# If you need a list of dictionaries instead
data = df.to_dict('records')

In [89]:
from reasoning.evaluator.math_grader import math_equal, extract_answer
math_equal('1','2-1')

True

In [90]:
def majority_vote(solutions):
    # extract the generated text for a single problem
    # print(f'found solutions: {solutions}')
    if len(solutions) == 1:
        return solutions[0]
    # create a matrix to store if any two solutions are equal
    equal_matrix = [[math_equal(solutions[i], solutions[j]) for j in range(i,len(solutions))] for i in range(len(solutions))]
    # majority vote: choose the solution whose number of equal solutions is the largest  (sum of each row)
    # if there are multiple solutions with the same number of equal solutions, choose the first one
    majority_solution = solutions[max(range(len(equal_matrix)), key=lambda i: sum(equal_matrix[i]))]
    return majority_solution

In [96]:
def weighted_majority_vote(solutions, rewards):
    # print(f'found solutions: {solutions}')
    # print(f'rewards: {rewards}')
    if len(solutions) == 1:
        return solutions[0]
        
    # Create a weighted vote counter for each solution
    weighted_votes = [0.0] * len(solutions)
    
    # For each solution, add its reward to all solutions that are equal to it
    for i in range(len(solutions)):
        for j in range(len(solutions)):
            if math_equal(solutions[i], solutions[j]):
                weighted_votes[i] += rewards[j]
    
    # print(f'weighted votes: {weighted_votes}')
    
    # Choose the solution with the highest weighted vote
    majority_solution = solutions[max(range(len(weighted_votes)), key=lambda i: weighted_votes[i])]
    return majority_solution

In [97]:
import random
repeat_count = 10
for _ in range(repeat_count):
    generated_solutions = []
    for question_idx in range(len(data)):
        temp = []
        for solution_idx in range(len(data[question_idx]['generated_solutions'])):
            temp.append(extract_answer(data[question_idx]['generated_solutions'][solution_idx]))
        # randomly sample 4 solutions
        if len(temp) >= 4:
            indices = random.sample(range(len(temp)), 4)
            sampled = [temp[i] for i in indices]
            sampled_rewards = [data[question_idx]['rewards'][i] for i in indices]
        else:
            sampled = temp  
        # result of majority vote
        generated_solutions.append(weighted_majority_vote(sampled, sampled_rewards))

    # now compare if the extracted answer is correct
    right_count = 0
    for question_idx in range(len(data)):
        right_count += math_equal(data[question_idx]['extracted_answer'], generated_solutions[question_idx])
    print(f'right count: {right_count}')

right count: 284
right count: 277
right count: 276
right count: 270
right count: 280
right count: 278
right count: 289
right count: 274
right count: 273
right count: 275


In [91]:
import random
repeat_count = 10
for _ in range(repeat_count):
    generated_solutions = []
    for question_idx in range(len(data)):
        temp = []
        for solution_idx in range(len(data[question_idx]['generated_solutions'])):
            temp.append(extract_answer(data[question_idx]['generated_solutions'][solution_idx]))
        # randomly sample 4 solutions
        if len(temp) >= 4:
            sampled = random.sample(temp, 4)
        else:
            sampled = temp  
        # result of majority vote
        generated_solutions.append(majority_vote(sampled))

    # now compare if the extracted answer is correct
    right_count = 0
    for question_idx in range(len(data)):
        right_count += math_equal(data[question_idx]['extracted_answer'], generated_solutions[question_idx])
    print(f'right count: {right_count}')


right count: 255
right count: 244
right count: 266
right count: 259
right count: 255
right count: 257
right count: 246
right count: 263
right count: 243
right count: 266
